# 离线评估测试 — 赛题一

调用 `eval_core.py` 对 `output/` 下的两个结果CSV进行离线评估。

**评估维度：**
- **Task1 (40%)**：聚类质量 — 轮廓系数 / CH指数 / Wasserstein距离 / DTW时序距离
- **Task2 (60%)**：分类F1 — 资金类型 + 交易意图（需真实标签）

**用法**：逐Cell运行即可。修改 `eval_core.py` 后，执行 **Cell 5** 重载模块。

In [ ]:
# Cell 1: 环境准备 & 导入模块
import os
import sys

# 智能定位项目根目录：如果 cwd 在 tests/ 下，则上一级为项目根
_cwd = os.path.abspath('.')
if os.path.basename(_cwd) == 'tests':
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    PROJECT_ROOT = _cwd

TESTS_DIR = os.path.join(PROJECT_ROOT, 'tests')

# 确保项目根目录和 tests 目录在 sys.path 中
for p in [PROJECT_ROOT, TESTS_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 导入评估核心模块
import eval_core

# ============================================================
# 路径配置（相对于项目根目录）
# ============================================================
FEATURES_DIR = os.path.join(PROJECT_ROOT, 'data', 'features')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', '100stock')
GROUND_TRUTH_PATH = None  # 有真实标签时改为 CSV 路径，如 'data/ground_truth.csv'

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FEATURES_DIR exists: {os.path.exists(FEATURES_DIR)}")
print(f"OUTPUT_DIR exists:   {os.path.exists(OUTPUT_DIR)}")
print(f"RAW_DIR exists:      {os.path.exists(RAW_DIR)}")
print("环境就绪 ✅")

PROJECT_ROOT: c:\Users\admin\Desktop\newdaba\tests
FEATURES_DIR exists: False
OUTPUT_DIR exists:   False
RAW_DIR exists:      False
环境就绪 ✅


In [2]:
# Cell 2: 运行完整评估
result = eval_core.run(
    features_dir=FEATURES_DIR,
    output_dir=OUTPUT_DIR,
    raw_dir=RAW_DIR,
    ground_truth_path=GROUND_TRUTH_PATH,
)

  离线评估开始

[1/5] 加载数据...


FileNotFoundError: [WinError 3] 系统找不到指定的路径。: 'c:\\Users\\admin\\Desktop\\newdaba\\tests\\data\\features'

In [ ]:
# Cell 3: Task1 详细结果 — 聚类质量
import pandas as pd

t1 = result['task1']

rows = []
for metric_name, metric_data in t1.items():
    if isinstance(metric_data, dict):
        row = {'指标': metric_name}
        for k, v in metric_data.items():
            if isinstance(v, float):
                row[k] = round(v, 4)
            else:
                row[k] = str(v)
        rows.append(row)

df_t1 = pd.DataFrame(rows)
df_t1 = df_t1[['指标', 'status'] + [c for c in df_t1.columns if c not in ['指标', 'status']]]
print("=" * 50)
print("  Task1 聚类质量指标")
print("=" * 50)
display(df_t1)

t1_score = result['task1_score']
print(f"\n📊 Task1 综合分: {t1_score['task1_score']:.2f} / 100")
print(f"   各维度得分: {t1_score['task1_details']}")

In [ ]:
# Cell 4: Task2 详细结果 — 分类分布 & 合规校验
import pandas as pd

t2 = result['task2']
dist = t2['distribution']

print("=" * 50)
print("  Task2 预测分布分析")
print("=" * 50)

# 资金类型分布
cap_df = pd.DataFrame({
    '资金类型': list(dist['capital_distribution'].keys()),
    '数量': list(dist['capital_distribution'].values()),
})
cap_df['占比'] = (cap_df['数量'] / cap_df['数量'].sum() * 100).round(1).astype(str) + '%'
print("\n📌 资金类型分布:")
display(cap_df)

# 交易意图分布
int_df = pd.DataFrame({
    '交易意图': list(dist['intention_distribution'].keys()),
    '数量': list(dist['intention_distribution'].values()),
})
int_df['占比'] = (int_df['数量'] / int_df['数量'].sum() * 100).round(1).astype(str) + '%'
print("\n📌 交易意图分布:")
display(int_df)

# 合规校验
print("\n📌 合规校验:")
print(f"  capital_type 合法值 [游资/量化/散户]: {'✅ 通过' if dist['capital_type_合规'] else '❌ 失败 (' + str(dist['invalid_capital_count']) + '条异常)'}")
print(f"  capital_intention 合法值 [买入/卖出/T0交易]: {'✅ 通过' if dist['capital_intention_合规'] else '❌ 失败 (' + str(dist['invalid_intention_count']) + '条异常)'}")

# F1 结果（如有真值）
if t2.get('f1'):
    f1r = t2['f1']
    print(f"\n📌 F1 Score (加权):")
    print(f"  资金类型 F1: {f1r['capital_f1']:.2f}%  (P={f1r['capital_precision']:.2f}%, R={f1r['capital_recall']:.2f}%)")
    print(f"  交易意图 F1: {f1r['intention_f1']:.2f}%  (P={f1r['intention_precision']:.2f}%, R={f1r['intention_recall']:.2f}%)")
    print(f"  匹配样本数: {f1r['matched_samples']}")
else:
    print(f"\n📌 F1 Score: 无真实标签，无法计算（设置 GROUND_TRUTH_PATH 后可计算）")

if t2.get('task2_score'):
    print(f"\n📊 Task2 综合分: {t2['task2_score']:.2f} / 100")

In [ ]:
# Cell 5: 动态重载 — 修改 eval_core.py 后运行此Cell即可生效，无需重启Kernel
import importlib
importlib.reload(eval_core)
print("eval_core 已重新加载 ✅")
print("请回到 Cell 2 重新运行评估。")

## 指标解读参考

| 指标 | 优秀 | 合格 | 需改进 |
|------|:----:|:----:|:------:|
| **轮廓系数** | > 0.5 | > 0.2 | ≤ 0.2 → 簇间重叠严重 |
| **CH指数** | > 100 | > 10 | ≤ 10 → 特征区分度不足 |
| **Wasserstein** | > 0.5 | > 0.1 | ≤ 0.1 → 簇间分布几乎相同 |
| **DTW ratio** | > 3.0 | > 1.5 | ≤ 1.5 → 时序形态缺乏区分 |
| **F1 (资金类型)** | > 70% | > 50% | ≤ 50% → 规则需优化 |
| **F1 (交易意图)** | > 70% | > 50% | ≤ 50% → 规则需优化 |

> **A/B榜总分 = Task1分 × 0.4 + Task2分 × 0.6**